# RAG Benchmark — Ingestion avec MinerU (magic-pdf)

Ce notebook permet d'exécuter l'ingestion des documents PDF dans Google Colab avec GPU.

**Pipeline :**
1. Installation des dépendances (MinerU, sentence-transformers, FAISS, etc.)
2. Clonage du repo GitHub
3. Upload des PDFs
4. Extraction via MinerU (tableaux complexes, images, structure)
5. Chunking (RecursiveCharacterTextSplitter)
6. Embeddings (sentence-transformers)
7. Indexation FAISS + BM25
8. Téléchargement du vector store

## 0 — Vérification GPU

In [ ]:
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 — Installation des dépendances

In [ ]:
!pip install -q "magic-pdf[full]>=0.10.0"
!pip install -q sentence-transformers faiss-cpu rank-bm25
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q pyyaml loguru

In [ ]:
import subprocess, shutil

# Vérifier que magic-pdf est bien installé
r = subprocess.run(["magic-pdf", "--version"], capture_output=True, text=True)
if r.returncode == 0:
    print(f"✓ magic-pdf installé : {r.stdout.strip() or r.stderr.strip()}")
else:
    print("⚠ magic-pdf introuvable, vérifiez l'installation")

# Vérifier la disponibilité GPU pour MinerU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device MinerU utilisera : {device}")

In [ ]:
import os, json, torch
from huggingface_hub import snapshot_download

DOWNLOAD_DIR = "/root/models/MinerU"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}\n")

print("Téléchargement pdf-extract-kit-1.0 ...")
snapshot_download(
    repo_id="opendatalab/pdf-extract-kit-1.0",
    local_dir=DOWNLOAD_DIR,
    ignore_patterns=["*.git*"],
)
print("✓ pdf-extract-kit-1.0 téléchargé")

print("Téléchargement layoutreader ...")
layoutreader_dir = os.path.join(DOWNLOAD_DIR, "models", "layoutreader")
snapshot_download(
    repo_id="hantian/layoutreader",
    local_dir=layoutreader_dir,
    ignore_patterns=["*.git*"],
)
print("✓ layoutreader téléchargé")

# Le vrai dossier de modèles est dans <DOWNLOAD_DIR>/models/
MODELS_DIR = os.path.join(DOWNLOAD_DIR, "models")

# Vérification : le fichier clé doit exister
check_file = os.path.join(MODELS_DIR, "MFD", "YOLO", "yolo_v8_ft.pt")
if os.path.exists(check_file):
    print(f"\n✓ Modèle YOLO trouvé : {check_file}")
else:
    for root, dirs, files in os.walk(DOWNLOAD_DIR):
        for f in files[:5]:
            print(f"  {os.path.join(root, f)}")
    raise FileNotFoundError(f"Modèle manquant : {check_file}")

# Créer magic-pdf.json
# NOTE: on utilise doclayout_yolo au lieu de layoutlmv3 pour éviter la dépendance detectron2
config = {
    "bucket_info": {
        "bucket-name-1": {"ak": "", "sk": "", "endpoint": ""},
        "bucket-name-2": {"ak": "", "sk": "", "endpoint": ""}
    },
    "temp-output-dir": "/tmp",
    "models-dir": MODELS_DIR,
    "layoutreader-model-dir": layoutreader_dir,
    "device-mode": device,
    "layout-config": {"model": "doclayout_yolo"},
    "formula-config": {
        "mfd_model": "yolo_v8_mfd",
        "mfr_model": "unimernet_small",
        "enable": True
    },
    "table-config": {
        "model": "rapid_table",
        "enable": True,
        "sub_model": "slanet_plus"
    },
    "lang": []
}
config_path = os.path.expanduser("~/magic-pdf.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"\n✓ magic-pdf.json créé : {config_path}")
print(f"  models-dir   : {config['models-dir']}")
print(f"  layout-model : {config['layout-config']['model']}")
print(f"  device-mode  : {config['device-mode']}")

## 2 — Cloner le repo et préparer l'environnement

In [ ]:
import os

REPO_URL = "https://github.com/Abdelmajid-BARANI/rag-classique-project.git"
REPO_DIR = "/content/rag-classique-project"

# Revenir à /content avant de supprimer (évite getcwd error)
os.chdir("/content")

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

!git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(f"Répertoire de travail : {os.getcwd()}")

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, REPO_DIR)

## 3 — Upload des fichiers PDF

Uploadez vos fichiers PDF contenant les données (y compris tableaux complexes).

In [ ]:
from google.colab import files
from pathlib import Path

# Créer le répertoire des données
DATA_DIR = os.path.join(REPO_DIR, "donnees rag")
os.makedirs(DATA_DIR, exist_ok=True)

print("Uploadez vos fichiers PDF :")
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = os.path.join(DATA_DIR, filename)
    with open(dest, 'wb') as f:
        f.write(content)
    print(f"  ✓ {filename} ({len(content)/1024:.1f} KB)")

pdf_files = list(Path(DATA_DIR).glob('*.pdf'))
print(f"\nTotal : {len(pdf_files)} fichier(s) PDF prêt(s)")

## 4 — Extraction PDF via MinerU

MinerU extrait le contenu en **Markdown** avec tables préservées en format markdown.

In [ ]:
# Désactiver OCR si modèles PaddleOCR manquants
# (non requis pour PDFs texte natif — layout + tables + formules restent actifs)
import os, inspect
import magic_pdf.model.pdf_extract_kit as pek

ocr_model = "/root/models/MinerU/models/OCR/paddleocr_torch/ch_PP-OCRv3_det_infer.pth"

if not os.path.exists(ocr_model):
    print("⚠ Modèles PaddleOCR absents → désactivation OCR")
    print("  (layout, tables et formules restent actifs)\n")

    # Trouver dynamiquement la classe qui a 'apply_ocr' dans son __init__
    patched = False
    for name, cls in inspect.getmembers(pek, inspect.isclass):
        try:
            sig = inspect.signature(cls.__init__)
            if 'apply_ocr' in sig.parameters:
                _orig = cls.__init__
                def _make_patch(orig):
                    def _patched_init(self, *a, **kw):
                        kw['apply_ocr'] = False
                        orig(self, *a, **kw)
                    return _patched_init
                cls.__init__ = _make_patch(_orig)
                print(f"✓ OCR désactivé (patch sur {name})")
                patched = True
                break
        except (ValueError, TypeError):
            continue

    if not patched:
        print("⚠ Classe non trouvée — tentative via doc_analyze")
        from magic_pdf.model import doc_analyze_by_custom_model as dam
        _orig_analyze = dam.doc_analyze
        def _no_ocr_analyze(*a, **kw):
            kw['apply_ocr'] = False
            return _orig_analyze(*a, **kw)
        dam.doc_analyze = _no_ocr_analyze
        print("✓ OCR désactivé (patch sur doc_analyze)")
else:
    print("✓ Modèles OCR disponibles")

In [ ]:
from ingestion.document_loader import DocumentLoader

# Répertoire de sortie MinerU (pour conserver les résultats intermédiaires)
MINERU_OUTPUT = os.path.join(REPO_DIR, "data", "mineru_output")
os.makedirs(MINERU_OUTPUT, exist_ok=True)

loader = DocumentLoader(DATA_DIR)
documents = loader.load_all_pdfs(output_dir=MINERU_OUTPUT)

print(f"\n{'='*60}")
print(f"Documents extraits : {len(documents)}")
for doc in documents:
    print(f"  - {doc['filename']} : {len(doc['content'])} caractères")

stats = loader.get_document_stats(documents)
print(f"\nStatistiques : {stats}")

In [ ]:
# Aperçu du contenu extrait (premier document, 2000 premiers caractères)
if documents:
    print(f"=== Aperçu : {documents[0]['filename']} ===")
    print(documents[0]['content'][:2000])

## 5 — Chunking

In [ ]:
from utils.helpers import load_config

config = load_config(os.path.join(REPO_DIR, "config.yaml"))

chunk_size = config.get("ingestion", {}).get("chunk_size", 1000)
chunk_overlap = config.get("ingestion", {}).get("chunk_overlap", 200)

print(f"chunk_size = {chunk_size}, chunk_overlap = {chunk_overlap}")

In [ ]:
from ingestion.chunker import DocumentChunker

chunker = DocumentChunker(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

chunks = chunker.chunk_documents(documents)

chunk_stats = chunker.get_chunk_stats(chunks)
print(f"\nStatistiques des chunks : {chunk_stats}")

In [ ]:
# Aperçu de quelques chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ({chunk['n_chars']} chars) ---")
    print(chunk['text'][:300])
    print("...")

## 6 — Génération des Embeddings

In [ ]:
from ingestion.embedder import BERTEmbedder

embedding_config = config.get("embeddings", {})
model_name = embedding_config.get("model_name", "paraphrase-multilingual-mpnet-base-v2")

# Utiliser le GPU si disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Modèle : {model_name}")
print(f"Device : {device}")

embedder = BERTEmbedder(model_name=model_name, device=device)
enriched_chunks = embedder.embed_chunks(chunks, batch_size=64)

print(f"\n✓ Embeddings générés pour {len(enriched_chunks)} chunks")
print(f"  Dimension : {embedder.get_embedding_dimension()}")

## 7 — Indexation FAISS + BM25

In [ ]:
from retrieval.vector_store import FAISSVectorStore

vector_store_config = config.get("vector_store", {})
persist_dir = os.path.join(REPO_DIR, "data", "vector_store")
os.makedirs(persist_dir, exist_ok=True)

vector_store = FAISSVectorStore(
    embedding_dim=embedder.get_embedding_dimension(),
    persist_directory=persist_dir
)

vector_store.add_chunks(enriched_chunks)
vector_store.save()

final_stats = vector_store.get_stats()
print(f"\n✓ Vector store créé et sauvegardé")
print(f"  Chunks indexés : {final_stats['total_chunks']}")
print(f"  Vecteurs FAISS : {final_stats['total_vectors']}")
print(f"  Dimension : {final_stats['embedding_dimension']}")

## 8 — Test de recherche rapide

In [ ]:
# Test avec une requête
test_query = "Quels sont les articles concernant les obligations contractuelles ?"

query_embedding = embedder.embed_text(test_query)

# Recherche hybride
retrieval_config = config.get("retrieval", {})
results = vector_store.hybrid_search(
    query_embedding=query_embedding,
    query_text=test_query,
    top_k=retrieval_config.get("top_k", 5),
    alpha=retrieval_config.get("hybrid_alpha", 0.5),
)

print(f"Requête : {test_query}")
print(f"Résultats : {len(results)}\n")
for r in results[:3]:
    print(f"[Score: {r.get('hybrid_score', r.get('score', 0)):.4f}] {r['text'][:200]}...\n")

## 9 — Télécharger le vector store

Téléchargez les fichiers FAISS pour les utiliser en local avec l'API.

In [ ]:
import shutil

# Créer une archive du vector store
archive_path = shutil.make_archive(
    base_name="/content/vector_store",
    format="zip",
    root_dir=os.path.join(REPO_DIR, "data"),
    base_dir="vector_store"
)

print(f"Archive créée : {archive_path}")
print(f"Taille : {os.path.getsize(archive_path) / 1024:.1f} KB")

# Télécharger
files.download(archive_path)
print("\n✓ Téléchargement lancé. Décompressez dans data/vector_store/ en local.")